In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import pandas as pd
import numpy as np
import os
import torch
import pickle
import config


In [3]:
from sentence_transformers import CrossEncoder,InputExample
from torch.utils.data import DataLoader
from src.metric import model_evaluation


In [4]:
torch.set_float32_matmul_precision("high")

In [5]:
np.random.seed(config.SEED)
torch.manual_seed(config.SEED)
torch.cuda.manual_seed_all(config.SEED)

In [6]:
path=config.CLEANED_DATA_DIR

In [7]:
with open(os.path.join(path,'train_df.pkl'),'rb') as f:
    train_df=pickle.load(f)
    
with open(os.path.join(path,'val_df.pkl'),'rb') as f:
    val_df=pickle.load(f)
        
with open(os.path.join(path,'test_df.pkl'),'rb') as f:
    test_df=pickle.load(f)
    


In [8]:
BATCH_SIZE=config.BASELINE_BATCH_SIZE

In [9]:

cross_train_examples=[
    InputExample(texts=[j,r],label=float(config.label_to_score[l]))
    for r,j,l in zip(train_df['resume_text'],train_df['job_description_text'],train_df['label'])
]

print(f"total training example:{len(cross_train_examples)}")

total training example:6240


In [10]:
g = torch.Generator()
g.manual_seed(config.SEED)

In [11]:
cross_train_dataloader=DataLoader(cross_train_examples,shuffle=True,batch_size=BATCH_SIZE)

In [12]:
cross_encoder_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2',num_labels=1,device=config.device)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [13]:
test_pairs=list(zip(test_df['job_description_text'],test_df['resume_text']))

scores=cross_encoder_model.predict(test_pairs,batch_size=64,show_progress_bar=False)

metrics=model_evaluation(scores,test_df,'job_description_text')

print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])


Spearman: 0.15692132526881591
Top-3 Accuracy: 0.9285714285714286
NDCG: 0.5601103442081681
MAP: 0.6565044443862946
MRR: 0.7311475409836066


In [14]:
model_save_path=os.path.join(config.BASELINE_MODEL_DIR,'cross_encoder')
os.makedirs(config.BASELINE_MODEL_DIR,exist_ok=True)

In [15]:
epochs=3
best_score=float('-inf')
min_delta=0.01
patience=2
count=0

for epoch in range(1,epochs+1):
    print(f"Epoch: {epoch}----------")

    cross_encoder_model.fit(train_dataloader=cross_train_dataloader,epochs=1,
                             warmup_steps=int(len(cross_train_dataloader) * epochs * 0.1),
                            show_progress_bar=True)
    
    val_pairs=list(zip(val_df['job_description_text'],val_df['resume_text']))

    scores=cross_encoder_model.predict(val_pairs,batch_size=64,show_progress_bar=False)
    
    metrics=model_evaluation(scores,val_df,'job_description_text')

    print("NDCG:",metrics['ndcg_val'])
    print("MAP:",metrics['map_score'])    

    final_score = 0.6*metrics['ndcg_val']+0.3*metrics['map_score']+0.1*metrics['mrr_score']
    
    if final_score>best_score + min_delta:
        best_score=final_score
        cross_encoder_model.save(model_save_path)
        count=0
    else:
        count+=1
        
    if count==patience:
        print("Early Stopping")
        break
    


Epoch: 1----------


Step,Training Loss


NDCG: 0.7123641507297579
MAP: 0.7900926069657945


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch: 2----------


Step,Training Loss


NDCG: 0.7147837311316968
MAP: 0.7923057114220425
Epoch: 3----------


Step,Training Loss


NDCG: 0.7319511392003718
MAP: 0.8023636799818604


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [16]:
cross_encoder_model=CrossEncoder(model_save_path,device=config.device)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [17]:
val_pairs=list(zip(val_df['job_description_text'],val_df['resume_text']))

scores=cross_encoder_model.predict(val_pairs,batch_size=64,show_progress_bar=False)


In [18]:
metrics=model_evaluation(scores,val_df,'job_description_text')

print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])

Spearman: 0.4686277546987827
Top-3 Accuracy: 1.0
NDCG: 0.7319511392003718
MAP: 0.8023636799818604
MRR: 0.8671875


In [19]:
ranked_result=[]
eval_df=val_df.copy()
eval_df['score']=scores

In [20]:
print("\nScore spread within groups:")
spreads = []
for jd, group in eval_df.groupby('job_description_text'):
    if len(group) > 1:
        spreads.append(group['score'].max() - group['score'].min())

print(f"Mean spread: {np.mean(spreads):.3f}")
print(f"% groups with spread < 0.1:"f"{(np.array(spreads) < 0.1).mean():.3f}")


Score spread within groups:
Mean spread: 3.512
% groups with spread < 0.1:0.000


In [21]:
for jd,group in eval_df.groupby('job_description_text'):
    ranked_group=group.sort_values("score",ascending=False)
    ranked_result.append(ranked_group)

final_rank_df=pd.concat(ranked_result)

In [22]:
i=0
for jd,group in final_rank_df.groupby('job_description_text'):
    if(len(group)>2 and len(group)<10):
        print("Job Description:\n",jd[:300])
        print(group[['label','score']])
        i+=1
        if i==3:
            break

Job Description:
 About Chamberlain Group:
Chamberlain Group is a global leader in access solutions. Our leading brands like LiftMaster, Chamberlain, Merlin and Grifco are found in millions of homes and commercial applications across the globe. Our innovative products powered by the myQ digital ecosystem provide cust
      label     score
1349      0 -1.205519
939       0 -1.379326
518       0 -1.379828
1326      0 -1.719135
442       0 -1.776312
3022      0 -2.115587
1833      0 -2.660353
3684      1 -2.990935
Job Description:
 About Hallgate Management: Hallgate Management is a property management company with a strong commitment to providing exceptional service to our clients and residents. We pride ourselves on our dedication to excellence, integrity, and continuous growth.
Position Overview: We are seeking a detail-ori
      label     score
401       0 -1.282691
300       0 -2.305629
1309      0 -2.607007
1129      0 -2.727800
894       0 -3.016538
Job Description:
 About Us Skadd

In [23]:
test_pairs=list(zip(test_df['job_description_text'],test_df['resume_text']))

scores=cross_encoder_model.predict(test_pairs,batch_size=64,show_progress_bar=False)

metrics=model_evaluation(scores,test_df,'job_description_text')

print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])


Spearman: 0.28286473991719735
Top-3 Accuracy: 0.9642857142857143
NDCG: 0.6264177734848825
MAP: 0.7266144282645547
MRR: 0.8495901639344263
